# Fig. 6: differential expression, GO enrichment, gene length and reversal of the aging signature

Works on a bulk RNA-seq gene count matrix (for example `GSE338230_counts_gene_all.csv`, or the output of
`salmon_quantification.ipynb`) and a sample sheet: DESeq2 for each contrast, preranked GSEA of GO
Biological Process 2023, and a per-gene youthful score (YS) that measures how much of the
age-associated change a treatment reverses, tested against an exact permutation null.

**Input:** a gene count matrix (Ensembl gene IDs in the first column, one column per sample) and a CSV
with one row per sample: the sample name, its condition and any covariates.

**Output** (in `OUT_DIR`): DESeq2 results per contrast, GSEA results, per-gene YS and permutation results
(CSV), and the plots (PDF).

In [ ]:
"""Install DESeq2 and GSEA."""
%pip install -q pydeseq2 gseapy

In [ ]:
"""Settings."""
from pathlib import Path

COUNTS = Path("GSE338230_counts_gene_all.csv")   # genes x samples
SAMPLES = Path("samples.csv")                     # one row per sample
OUT_DIR = Path("results/bulk_rnaseq")

SAMPLE_KEY = "sample"         # sample-sheet column holding the count-matrix column names
CONDITION_KEY = "condition"   # sample-sheet column compared in differential expression
COVARIATES = []               # further sample-sheet columns in the design, e.g. ["sex"]
CONTRASTS = []                # (test, reference) values of CONDITION_KEY, e.g. [("B", "A")]

AGING = None                  # (aged, young) values of CONDITION_KEY: the aging signature
TREATMENTS = []               # values of CONDITION_KEY scored against the aged group
N_JOBS = -1                   # parallel jobs for the permutation null (-1 = all cores)

In [ ]:
"""Analysis parameters (values used in the paper)."""
CFG = dict(
    # DESeq2
    min_total_counts=10,      # per gene, summed over all samples (condition-blind)
    alpha=0.1,                # independent-filtering target (DESeq2 default)

    # preranked GSEA
    library="GO_Biological_Process_2023",   # Enrichr library (human symbols)
    min_size=15, max_size=500, n_perm=1000, seed=42,
    gtf="https://ftp.ensembl.org/pub/release-109/gtf/mus_musculus/Mus_musculus.GRCm39.109.gtf.gz",  # gene symbols
    fdr=0.05,                 # significance for the dot plot and the table of terms
    plot_max_size=100,        # dot plot: largest gene set shown (measured genes), leaves out umbrella terms

    # youthful score
    sig_padj=0.05, sig_lfc=0.3,   # age-regulated genes: aging-contrast padj and |log2FC|
    max_splits=None,              # permutation splits evaluated; None = all (exact test)
)

In [ ]:
"""Imports."""
import gzip, itertools, re, urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from joblib import Parallel, delayed
from scipy.stats import spearmanr
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference
import gseapy as gp

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
"""Library versions used in this run."""
import sys, platform
from importlib.metadata import version, PackageNotFoundError

print(f"{'python':<12}{sys.version.split()[0]} ({platform.platform()})")
for pkg in ["numpy", "pandas", "scipy", "matplotlib", "joblib", "pydeseq2", "gseapy"]:
    try:
        print(f"{pkg:<12}{version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<12}not installed")

## Functions

In [ ]:
"""Plot style."""
mpl.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    "font.size": 7, "axes.labelsize": 7, "axes.titlesize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2, "ytick.major.size": 2,
    "axes.spines.top": False, "axes.spines.right": False,
})
MM = 1 / 25.4
DIVERGING = LinearSegmentedColormap.from_list("nes", ["#1c5cab", "#6da7ec", "#f0efec", "#ec8a7c", "#b8312f"])

In [ ]:
"""DESeq2, youthful score, gene symbols."""
DESIGN = "~" + " + ".join(list(COVARIATES) + [CONDITION_KEY])
BINS = np.arange(-3, 3.0001, 0.05)   # per-gene YS histograms (values clipped to +-3)


def tag(s):
    return re.sub(r"[^A-Za-z0-9]+", "-", str(s)).strip("-")


def fit(counts, meta, n_cpus=None):
    """DESeq2: median-of-ratios size factors, dispersion shrinkage, refit of Cook's outliers."""
    inf = DefaultInference(n_cpus=n_cpus)
    dds = DeseqDataSet(counts=counts.T, metadata=meta, design=DESIGN, refit_cooks=True,
                       inference=inf, quiet=True)
    dds.deseq2()
    return dds, inf


def results(dds, inf, test, ref):
    """Wald test of one contrast, BH-adjusted with independent and Cook's filtering."""
    st = DeseqStats(dds, contrast=[CONDITION_KEY, test, ref], alpha=CFG["alpha"], inference=inf, quiet=True)
    st.summary()
    return st.results_df


def youthful_score(age, trt):
    """YS = -log2FC(treatment vs aged) / log2FC(aged vs young), for the age-regulated genes."""
    sel = (age.padj < CFG["sig_padj"]) & (age.log2FoldChange.abs() >= CFG["sig_lfc"])
    ys = -trt.log2FoldChange.reindex(age.index)[sel] / age.log2FoldChange[sel]
    return ys[np.isfinite(ys)]


def split_ys(counts, meta, pool, aged, trt):
    """YS after labelling the samples in `aged` as aged and the rest of `pool` as `trt` (full refit)."""
    m = meta.copy()
    m.loc[pool, CONDITION_KEY] = trt
    m.loc[list(aged), CONDITION_KEY] = AGING[0]
    dds, inf = fit(counts, m, n_cpus=1)
    return youthful_score(results(dds, inf, *AGING), results(dds, inf, trt, AGING[0]))


def gene_table(gtf):
    """Gene name and exon-union length (bp) per Ensembl gene ID, from a GTF (path or URL; a URL is
    downloaded once into OUT_DIR)."""
    path = Path(gtf)
    if str(gtf).startswith(("http://", "https://", "ftp://")):
        path = OUT_DIR / str(gtf).rsplit("/", 1)[-1]
        if not path.exists():
            urllib.request.urlretrieve(gtf, path)
    sym, exons = {}, {}
    with (gzip.open(path, "rt") if path.suffix == ".gz" else open(path)) as fh:
        for line in fh:
            f = line.split("\t", 8)
            if len(f) < 9 or f[2] not in ("gene", "exon"):
                continue
            gid = re.search(r'gene_id "([^"]+)"', f[8]).group(1)
            if f[2] == "gene":
                name = re.search(r'gene_name "([^"]+)"', f[8])
                sym[gid] = name.group(1) if name else gid
            else:
                exons.setdefault(gid, []).append((int(f[3]), int(f[4])))
    length = {}
    for gid, iv in exons.items():          # union of the exons of all transcripts of the gene
        iv.sort()
        total, (s0, e0) = 0, iv[0]
        for st, en in iv[1:]:
            if st > e0:
                total, s0, e0 = total + e0 - s0 + 1, st, en
            else:
                e0 = max(e0, en)
        length[gid] = total + e0 - s0 + 1
    return pd.DataFrame({"symbol": pd.Series(sym), "length": pd.Series(length)})

## Load

In [ ]:
"""Counts and sample sheet; genes with too few reads are dropped (condition-blind)."""
meta = pd.read_csv(SAMPLES, dtype={SAMPLE_KEY: str, CONDITION_KEY: str}).set_index(SAMPLE_KEY)
counts = pd.read_csv(COUNTS, index_col=0)[meta.index].round().astype(int)
counts = counts[counts.sum(axis=1) >= CFG["min_total_counts"]]
print(f"{counts.shape[1]} samples, {counts.shape[0]} genes kept")
print(meta[CONDITION_KEY].value_counts().to_string())

## Differential expression

DESeq2 (pydeseq2) with the condition and covariates in the design: median-of-ratios size factors,
dispersion shrinkage and the Wald test, Benjamini-Hochberg adjustment with independent filtering and
Cook's-distance filtering, as in DESeq2.

In [ ]:
"""DESeq2 fit and the Wald test of every contrast."""
dds, inf = fit(counts, meta)
de_dir = OUT_DIR / "deseq2"
de_dir.mkdir(exist_ok=True)
RESULTS, summary = {}, []
for a, b in CONTRASTS:
    res = results(dds, inf, a, b)
    res.to_csv(de_dir / f"{tag(a)}_vs_{tag(b)}.csv")
    RESULTS[(a, b)] = res
    sig = res.padj < 0.05
    summary.append(dict(contrast=f"{a} vs {b}", up=int((sig & (res.log2FoldChange > 0)).sum()),
                        down=int((sig & (res.log2FoldChange < 0)).sum())))
print(pd.DataFrame(summary).to_string(index=False))

## Gene set enrichment

Preranked GSEA (gseapy) of every contrast on the DESeq2 Wald statistic, against GO Biological Process 2023
(Enrichr). Gene symbols (from the Ensembl GTF) are upper-cased to match the library, the statistics of
duplicated symbols are averaged, and infinite statistics are set to the largest finite value of the same
sign. Gene sets with 15 to 500 measured genes are tested (1,000 permutations, seed 42). The dot plot shows
the terms of at most 100 genes with FDR < 0.05 in every contrast; `gsea_significant.csv` lists every term
with FDR < 0.05 in at least one.

In [ ]:
"""Preranked GSEA of every contrast."""
genes = gene_table(CFG["gtf"])
symbols = genes.symbol
library = gp.get_library(name=CFG["library"], organism="Human")
parts = []
for (a, b), res in RESULTS.items():
    st = res["stat"].dropna()
    st = st.groupby(symbols.reindex(st.index).fillna(pd.Series(st.index, index=st.index)).str.upper()).mean()
    fin = st[np.isfinite(st)]
    rnk = st.clip(fin.min(), fin.max()).sort_values(ascending=False)
    pre = gp.prerank(rnk=rnk, gene_sets=library, min_size=CFG["min_size"], max_size=CFG["max_size"],
                     permutation_num=CFG["n_perm"], seed=CFG["seed"], threads=4, outdir=None,
                     no_plot=True, verbose=False)
    r = pre.res2d.copy()
    r.insert(0, "contrast", f"{a} vs {b}")
    parts.append(r)
GSEA = pd.concat(parts, ignore_index=True)
for c in ("ES", "NES", "NOM p-val", "FDR q-val", "FWER p-val"):
    GSEA[c] = pd.to_numeric(GSEA[c], errors="coerce")
GSEA.to_csv(OUT_DIR / "gsea_prerank.csv", index=False)
print((GSEA["FDR q-val"] < CFG["fdr"]).groupby(GSEA.contrast, sort=False).sum().rename(f"sets at FDR < {CFG['fdr']}").to_string())

In [ ]:
"""Terms significant in every contrast (dot plot) and in any contrast (table)."""
TERMS = []   # [] = terms of <= CFG["plot_max_size"] genes with FDR < CFG["fdr"] in every contrast (the paper); or a list
RENAME = {   # shorter display names
    "Mitochondrial ATP Synthesis Coupled Electron Transport (GO:0042775)": "Mito. ATP synthesis-coupled electron transport",
    "Proton Motive Force-Driven ATP Synthesis (GO:0015986)": "PMF-driven ATP synthesis",
    "Proton Motive Force-Driven Mitochondrial ATP Synthesis (GO:0042776)": "PMF-driven mitochondrial ATP synthesis",
    "Mitochondrial Electron Transport, NADH To Ubiquinone (GO:0006120)": "Mito. electron transport, NADH to ubiquinone",
    "Mitochondrial Electron Transport, Cytochrome C To Oxygen (GO:0006123)": "Mito. electron transport, cytochrome c to O2",
    "Mitochondrial Respiratory Chain Complex Assembly (GO:0033108)": "Mito. respiratory chain complex assembly",
    "Mitochondrial Respiratory Chain Complex I Assembly (GO:0032981)": "Mito. respiratory chain complex I assembly",
}


def pretty(t):
    """Display name: GO ID dropped, sentence case (words such as mRNA, NADH or Golgi are kept)."""
    if t in RENAME:
        return RENAME[t]
    low = lambda x: "-".join(q.lower() if (q[:1].isupper() and q[1:].islower() and q != "Golgi") else q
                             for q in x.split("-"))
    w = re.sub(r"\s*\(GO:\d+\)$", "", t).split(" ")
    head = w[0].split("-")
    s = " ".join(["-".join([head[0]] + [low(q) for q in head[1:]])] + [low(x) for x in w[1:]])
    return s if len(s) <= 58 else s[:57].rstrip() + "..."


g = GSEA.dropna(subset=["NES"]).copy()
g["n_genes"] = g["Tag %"].str.split("/").str[1].astype(int)
cols = list(dict.fromkeys(g.contrast))
n_sig = g[g["FDR q-val"] < CFG["fdr"]].groupby("Term").contrast.nunique()

# table: every term significant in at least one contrast
wide = g.pivot(index="Term", columns="contrast", values=["NES", "FDR q-val"])
wide.columns = [f"{'NES' if a == 'NES' else 'FDR'} ({c})" for a, c in wide.columns]
wide = wide[[f"{k} ({c})" for c in cols for k in ("NES", "FDR")]]
wide.insert(0, "n_significant", n_sig.reindex(wide.index).fillna(0).astype(int))
wide = wide[wide.n_significant > 0].sort_values(["n_significant", f"NES ({cols[0]})"], ascending=False)
wide.to_csv(OUT_DIR / "gsea_significant.csv")
print(f"{len(wide)} terms with FDR < {CFG['fdr']} in at least one contrast, {int((wide.n_significant == len(cols)).sum())} in all")

# dot plot, rows ordered by the NES of the first contrast; a grey ring marks FDR >= CFG["fdr"]
small = g[(g["FDR q-val"] < CFG["fdr"]) & (g.n_genes <= CFG["plot_max_size"])].groupby("Term").contrast.nunique()
terms = TERMS or list(small[small == len(cols)].index)
terms = list(g[(g.contrast == cols[0]) & g.Term.isin(terms)].sort_values("NES", ascending=False).Term)
size = lambda q: 8 + 34 * min(-np.log10(max(q, 1e-3)), 3) / 3
norm = TwoSlopeNorm(vmin=-3, vcenter=0, vmax=3)
fig, ax = plt.subplots(figsize=((70 + 18 * len(cols)) * MM, (14 + 4 * len(terms)) * MM))
for _, r in g[g.Term.isin(terms)].iterrows():
    sig = r["FDR q-val"] < CFG["fdr"]
    ax.scatter(cols.index(r.contrast), terms.index(r.Term), s=size(r["FDR q-val"]), c=[r.NES], cmap=DIVERGING,
               norm=norm, edgecolors="0.1" if sig else "0.65", linewidths=0.6 if sig else 0.4, zorder=3)
ax.set_xticks(range(len(cols)), [c.replace(" vs ", "\nvs ") for c in cols])
ax.set_yticks(range(len(terms)), [pretty(t) for t in terms])
ax.set_xlim(-0.6, len(cols) - 0.4)
ax.set_ylim(len(terms) - 0.5, -0.6)
ax.xaxis.tick_top()
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.grid(axis="y", color="0.9", lw=0.4)
ax.set_axisbelow(True)
fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=DIVERGING), ax=ax, shrink=0.4, label="NES")
for q in (0.05, 0.01, 0.001):
    ax.scatter([], [], s=size(q), facecolor="white", edgecolor="0.1", linewidths=0.6, label=f"{q:g}")
ax.legend(title="FDR", loc="upper left", bbox_to_anchor=(1.3, 0.45), frameon=False, labelspacing=1.0)
fig.savefig(OUT_DIR / "gsea_dotplot.pdf", bbox_inches="tight")
plt.show()

## Gene length

Aged tissues gain short, highly expressed transcripts relative to long ones (Stoeger et al., Nat Aging
2022). For each contrast, the Spearman correlation between log2 fold change and exon-union gene length is
computed over the genes with baseMean >= 10, and the mean log2 fold change of each gene set in
`LENGTH_SETS` is compared with 10,000 random gene sets matched for length and expression (for every
member, a gene from the same length-decile x expression-decile stratum; two-sided empirical P).

In [ ]:
"""Fold change against gene length, and gene sets against length- and expression-matched null sets."""
LENGTH_SETS = ["Oxidative Phosphorylation (GO:0006119)", "Cytoplasmic Translation (GO:0002181)"]
N_NULL = 10_000
rng = np.random.default_rng(CFG["seed"])
upper = genes.symbol.str.upper()
rows, nulls, deciles = [], [], []
for (a, b), res in RESULTS.items():
    con = f"{a} vs {b}"
    t = res[(res.baseMean >= 10) & res.log2FoldChange.notna()].join(genes.length, how="inner")
    t["len_bin"] = pd.qcut(np.log10(t.length), 10, labels=False)
    t["exp_bin"] = pd.qcut(np.log10(t.baseMean), 10, labels=False)
    deciles.append(t.groupby("len_bin").agg(length=("length", "median"), median_lfc=("log2FoldChange", "median"))
                   .assign(contrast=con))
    rho = spearmanr(t.log2FoldChange, t.length)[0]
    lfc, code = t.log2FoldChange.to_numpy(), (t.len_bin * 10 + t.exp_bin).to_numpy()
    for term in LENGTH_SETS:
        member = upper.reindex(t.index).isin(set(library[term])).to_numpy()
        need = pd.Series(code[member]).value_counts()
        pools = {k: np.flatnonzero((code == k) & ~member) for k in need.index}
        null = np.array([np.concatenate([lfc[rng.choice(pools[k], n, replace=len(pools[k]) < n)]
                                         for k, n in need.items()]).mean() for _ in range(N_NULL)])
        obs = lfc[member].mean()
        p = (np.sum(np.abs(null - null.mean()) >= abs(obs - null.mean())) + 1) / (N_NULL + 1)
        rows.append(dict(contrast=con, gene_set=term, n_genes=int(member.sum()), rho_lfc_length=rho,
                         mean_lfc=obs, null_mean=null.mean(), null_2_5=np.quantile(null, 0.025),
                         null_97_5=np.quantile(null, 0.975), P=p))
        nulls.append(pd.DataFrame({"contrast": con, "gene_set": term, "null_mean_lfc": null}))
LENGTH, DECILES = pd.DataFrame(rows), pd.concat(deciles)
LENGTH.to_csv(OUT_DIR / "gene_length.csv", index=False)
DECILES.to_csv(OUT_DIR / "gene_length_deciles.csv")
pd.concat(nulls).to_csv(OUT_DIR / "gene_length_null.csv", index=False)
print(LENGTH.round(3).to_string(index=False))

In [ ]:
"""Median log2FC by gene-length decile for every contrast (Fig. 6)."""
COLOURS = {}   # contrast -> colour; empty = default colours
fig, ax = plt.subplots(figsize=(60 * MM, 50 * MM))
for i, (con, d) in enumerate(DECILES.groupby("contrast", sort=False)):
    rho = LENGTH.loc[LENGTH.contrast == con, "rho_lfc_length"].iloc[0]
    ax.plot(d.length / 1000, d.median_lfc, marker="o", ms=3, lw=1.2, color=COLOURS.get(con, f"C{i}"),
            label=f"{con} (rho = {rho:.2f})")
ax.set_xscale("log")
ax.axhline(0, color="0.8", lw=0.6, zorder=0)
ax.set(xlabel="Gene length (kb, decile median)", ylabel="Median log2FC")
ax.legend(frameon=False, loc="lower left")
fig.savefig(OUT_DIR / "gene_length.pdf", bbox_inches="tight")
plt.show()

## Reversal of the aging signature

For each treatment, every age-regulated gene (aging contrast padj < 0.05 and |log2FC| >= 0.3) gets a
youthful score, YS = -log2FC(treatment vs aged) / log2FC(aged vs young): 1 is a full return to the
young level, 0 no change, < 0 a further change in the aging direction. The treatment effect is the median
YS.

Both fold changes use the aged group as reference, and the genes are selected on the aging contrast, so
sampling noise in the aged group alone makes the YS positive (regression to the mean). The null is
therefore built by enumerating every split of the aged and treated samples into two groups of the
original sizes and rerunning the whole pipeline (DESeq2 on all samples, aging contrast, gene selection,
YS) for each split. The exact one-sided P is the fraction of splits, the observed one included, whose
median YS is at least the observed value.

In [ ]:
"""Observed YS of each treatment and its exact permutation null."""
ys_dir = OUT_DIR / "youthful_score"
ys_dir.mkdir(exist_ok=True)
age = results(dds, inf, *AGING)
aged_obs = tuple(sorted(meta.index[meta[CONDITION_KEY] == AGING[0]]))
SIG = {}
for trt in TREATMENTS:
    ys = youthful_score(age, results(dds, inf, trt, AGING[0]))
    ys.rename("YS").to_csv(ys_dir / f"YS_{tag(trt)}.csv")
    pool = sorted(meta.index[meta[CONDITION_KEY].isin([AGING[0], trt])])
    splits = [aged_obs] + [s for s in itertools.combinations(pool, len(aged_obs)) if s != aged_obs]
    splits = splits[:CFG["max_splits"]] if CFG["max_splits"] else splits
    null = Parallel(n_jobs=N_JOBS)(delayed(split_ys)(counts, meta, pool, s, trt) for s in splits)
    perm = pd.DataFrame({"aged_samples": [";".join(s) for s in splits], "observed": [s == aged_obs for s in splits],
                         "median_YS": [r.median() for r in null], "n_genes": [len(r) for r in null]})
    perm.to_csv(ys_dir / f"permutations_{tag(trt)}.csv", index=False)
    obs = perm.median_YS[0]          # the observed split; equals ys.median()
    p = (perm.median_YS >= obs).mean()
    hist = sum(np.histogram(np.clip(r, -3, 3), bins=BINS)[0] for r in null[1:])
    SIG[trt] = dict(ys=ys, perm=perm, hist=hist, p=p)
    print(f"{trt}: {len(ys)} age-regulated genes, median YS {ys.median():.3f} "
          f"(IQR {ys.quantile(.25):.3f} to {ys.quantile(.75):.3f}); null mean {perm.median_YS[1:].mean():.3f}; "
          f"exact P = {p:.4f} ({len(perm)} splits)")

In [ ]:
"""Per-gene YS against the pooled null, and the null distribution of the median YS."""
COLOURS = {}        # treatment -> colour; empty = grey
XLIM = (-1.5, 2.0)  # per-gene YS axis
W = 0.025           # bin width of the median-YS histogram

ctr = (BINS[:-1] + BINS[1:]) / 2
fig, axes = plt.subplots(len(SIG), 2, figsize=(120 * MM, 38 * MM * len(SIG)), squeeze=False)
for i, (trt, d) in enumerate(SIG.items()):
    col = COLOURS.get(trt, "0.6")
    ax = axes[i, 0]
    inr = (BINS[:-1] >= XLIM[0] - 1e-9) & (BINS[1:] <= XLIM[1] + 1e-9)   # only the shown range is drawn
    w = np.diff(BINS)[0]
    obs = np.histogram(np.clip(d["ys"], -3, 3), bins=BINS)[0] / len(d["ys"]) / w   # density over all genes
    ax.bar(ctr[inr], obs[inr], width=w, color=col, label=trt)
    ax.plot(ctr[inr], (d["hist"] / d["hist"].sum() / w)[inr], color="0.25", lw=1.0, label="null")
    ax.axvline(0, color="0.6", lw=0.6, ls=":")
    ax.axvline(d["ys"].median(), color="0.1", lw=0.8)
    ax.set(xlim=XLIM, xlabel="Youthful score (YS)", ylabel="Density")
    ax.legend(loc="upper left", frameon=False)

    ax = axes[i, 1]
    m = d["perm"].median_YS
    edges = np.arange(np.floor(m.min() / W) * W, np.ceil(m.max() / W) * W + W, W)
    ax.hist(m[1:], bins=edges, color="0.78", edgecolor="white", linewidth=0.3)
    ax.axvline(m[0], color=COLOURS.get(trt, "0.1"), lw=1.4)
    ax.text(0.98, 0.95, f"{trt}\nobserved {m[0]:.2f}\nnull mean {m[1:].mean():.2f}\nexact P = {d['p']:.3f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=6)
    ax.set(xlabel="Median YS", ylabel="Splits")
fig.tight_layout()
fig.savefig(OUT_DIR / "youthful_score.pdf")
plt.show()